# Top-k Evaluator (MAD auf Testsplit)

Dieses Notebook evaluiert die Featureauswahl von `gnnexplainer`, `integrated_gradients`, `ablation`, `random` mit identischem Training-Setup wie im Hauptnotebook.

Primaere Vergleichsmetrik:
- `target=H` -> `MAD_H` auf Testsplit
- `target=C` -> `MAD_C` auf Testsplit


## Setup

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path


def _discover_local_project_root() -> Path:
    # Optional override for local runs
    env_root = os.environ.get("GNN4NMR_PROJECT_ROOT", "").strip()
    if env_root:
        cand = Path(env_root).expanduser().resolve()
        if (cand / "scripts").exists() and (cand / "notebooks" / "topk_evaluator_lib.py").exists():
            return cand
        raise FileNotFoundError(
            f"GNN4NMR_PROJECT_ROOT gesetzt, aber ungueltig: {cand} (erwarte scripts/ und notebooks/topk_evaluator_lib.py)"
        )

    cwd = Path.cwd().resolve()

    # Case A: current or parent is .../gnn4nmr
    for base in [cwd, *cwd.parents]:
        if (base / "scripts").exists() and (base / "notebooks" / "topk_evaluator_lib.py").exists():
            return base

    # Case B: current or parent is repo root containing gnn4nmr/
    for base in [cwd, *cwd.parents]:
        cand = base / "gnn4nmr"
        if (cand / "scripts").exists() and (cand / "notebooks" / "topk_evaluator_lib.py").exists():
            return cand

    raise FileNotFoundError(
        "Lokales Projekt nicht gefunden. Starte das Notebook aus dem gnn4nmr-Repo "
        "oder setze GNN4NMR_PROJECT_ROOT auf den Ordner, der scripts/ und notebooks/ enthaelt."
    )


PROJECT_ROOT = _discover_local_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

for pth in [str(NOTEBOOK_DIR), str(SCRIPTS_DIR), str(PROJECT_ROOT)]:
    if pth not in sys.path:
        sys.path.insert(0, pth)

from topk_evaluator_lib import (
    load_and_normalize_topk_csv,
    resolve_feature_selection,
    FeatureSelectShiftDataset,
    compute_test_mad,
    run_all_topk_experiments,
    build_feature_lookup,
    validate_topk_table,
    configure_runtime_for_a100,
    default_num_workers,
    load_results_for_visualization,
    plot_mad_distribution,
    plot_seed_stability,
    build_ranking_tables,
    plot_learning_curves,
)

# Default setup (aligned with gnn4nmr.ipynb)
DEFAULT_CONFIG = {
    "seed": 0,
    "batch_size": 2,
    "num_epochs": 80,
    "lr": 2e-4,
    "hidden_dim": 128,
    "out_dim": 128,
    "num_gnn_layers": 3,
    "operator_type": "GraphConv",
    "operator_kwargs": {},
    "encoder_dropout": 0.1,
    "gnnlayer_dropout": 0.1,
    "optimizer": "Adam",
    "weight_decay": 5e-5,
    "scheduler_factor": 0.7,
    "scheduler_patience": 15,
    "loss_weight_H": 10.0,
    "loss_weight_C": 1.0,
    "normalize_node_features": True,
    "normalize_edge_features": True,
    "split_ratio": (0.7, 0.15, 0.15),
    "data_file_name": "all_graphs_with_length_filtered.pkl",
}

TOPK_CSV_PATH = PROJECT_ROOT / "results" / "experiments" / "topk-feats" / "sparcity70.csv"
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_ROOT = PROJECT_ROOT / "results" / "experiments" / "topk-evaluator"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = RESULTS_ROOT / RUN_TIMESTAMP
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [100, 101, 103]
USE_WANDB = True
SKIP_COMPLETED = True
RUN_SMOKE_TEST = False
SMOKE_SEEDS = [0]
SMOKE_NUM_EPOCHS = 3

FAST_NUM_WORKERS = default_num_workers()
FAST_PIN_MEMORY = True
FAST_PREFETCH_FACTOR = 4
USE_AMP = True
AMP_DTYPE = "bf16"
ENABLE_TF32 = True
CUDNN_BENCHMARK = True
MATMUL_PRECISION = "high"
WANDB_LOG_EVERY_N_EPOCHS = 2
COMPILE_MODEL = False

FAST_BATCH_SIZE_OVERRIDE = None
if FAST_BATCH_SIZE_OVERRIDE is not None:
    DEFAULT_CONFIG["batch_size"] = int(FAST_BATCH_SIZE_OVERRIDE)

configure_runtime_for_a100(
    enable_tf32=ENABLE_TF32,
    cudnn_benchmark=CUDNN_BENCHMARK,
    matmul_precision=MATMUL_PRECISION,
)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"NOTEBOOK_DIR: {NOTEBOOK_DIR}")
print(f"TOPK_CSV_PATH: {TOPK_CSV_PATH}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"FAST_NUM_WORKERS: {FAST_NUM_WORKERS}, FAST_PIN_MEMORY: {FAST_PIN_MEMORY}, FAST_PREFETCH_FACTOR: {FAST_PREFETCH_FACTOR}")
print(f"USE_AMP: {USE_AMP} ({AMP_DTYPE}), TF32: {ENABLE_TF32}, CUDNN_BENCHMARK: {CUDNN_BENCHMARK}")
print(f"Effective batch size: {DEFAULT_CONFIG['batch_size']}")


In [ ]:
# Preflight checks
TOPK_DF = load_and_normalize_topk_csv(TOPK_CSV_PATH)
FEATURE_LOOKUP = build_feature_lookup()
DETERMINISTIC_SELECTIONS, RANDOM_POOL_BY_TARGET, RANDOM_K_BY_TARGET, PREFLIGHT_DF = validate_topk_table(
    TOPK_DF,
    FEATURE_LOOKUP,
)

display(PREFLIGHT_DF)
print(f"rows in normalized top-k table: {len(TOPK_DF)}")
print("random k by target:", RANDOM_K_BY_TARGET)

## Berechnung

In [ ]:
RUN_EXPERIMENTS = False

active_seeds = SMOKE_SEEDS if RUN_SMOKE_TEST else SEEDS
epoch_override = SMOKE_NUM_EPOCHS if RUN_SMOKE_TEST else None

if RUN_EXPERIMENTS:
    run_results_df, epoch_history_df, feature_sets_df, summary_df = run_all_topk_experiments(
        topk_csv_path=TOPK_CSV_PATH,
        output_dir=OUTPUT_DIR,
        seeds=active_seeds,
        base_config=DEFAULT_CONFIG,
        data_dir=DATA_DIR,
        use_wandb=USE_WANDB,
        skip_completed=SKIP_COMPLETED,
        smoke_num_epochs=epoch_override,
        num_workers=FAST_NUM_WORKERS,
        pin_memory=FAST_PIN_MEMORY,
        prefetch_factor=FAST_PREFETCH_FACTOR,
        use_amp=USE_AMP,
        amp_dtype=AMP_DTYPE,
        enable_tf32=ENABLE_TF32,
        cudnn_benchmark=CUDNN_BENCHMARK,
        matmul_precision=MATMUL_PRECISION,
        log_every_n_epochs=WANDB_LOG_EVERY_N_EPOCHS,
        compile_model=COMPILE_MODEL,
    )
    print("Experiment finished.")
    display(summary_df)
else:
    print("Set RUN_EXPERIMENTS=True to start training.")
    print(f"Planned runs: {len(active_seeds) * 4 * 2}")


## Visualisierung

In [ ]:
VIS_RUN_DIR = OUTPUT_DIR

run_df_vis, epoch_df_vis, feat_df_vis = load_results_for_visualization(VIS_RUN_DIR)

if run_df_vis.empty:
    print(f"No run_results.csv found in {VIS_RUN_DIR}")
else:
    print(f"Loaded run results: {len(run_df_vis)} rows")
    display(run_df_vis.head())

    plot_mad_distribution(run_df_vis)
    plot_seed_stability(run_df_vis)

    h_rank_df, c_rank_df = build_ranking_tables(run_df_vis)
    print("Ranking table target=H (lower MAD is better):")
    display(h_rank_df)

    print("Ranking table target=C (lower MAD is better):")
    display(c_rank_df)

    plot_learning_curves(epoch_df_vis)